Setup

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
len(documents)

72

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [19]:
from dotenv import load_dotenv
from openai import OpenAI
import json
from evaluation_utils import llm_structured

load_dotenv()
openai_client=OpenAI()

In [21]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [32]:
import pandas as pd

Q1

In [9]:
sel_pages=documents[:3]

In [14]:
sel_pages

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [27]:
recs=[]

In [28]:
recs

[]

In [29]:
recs = []
input_tokens = []

for i in sel_pages:
    user_prompt=json.dumps({
        "filename":i["filename"],
        "content":i["content"]
    }, indent=2)

    questions,usage = llm_structured(
        client=openai_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions
    )

    input_tokens.append(usage.input_tokens)

    for q in questions.questions:
        recs.append({
            "filename":i["filename"],
            "question":q
        })

In [34]:
avg_tokens = sum(input_tokens)/len(input_tokens)
avg_tokens

1357.0

Q2 - Used provided ground-truth.csv file

In [35]:
df_ground_truth = pd.read_csv('ground-truth.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [36]:
len(ground_truth)

360

In [37]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [38]:
len(chunks)

295

In [41]:
from minsearch import Index, VectorSearch

In [42]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [43]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [44]:
q = ground_truth[0]["question"]

In [46]:
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [47]:
t_results=text_search(q)
t_results

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [48]:
t_results[0]["filename"]

'01-agentic-rag/lessons/03-rag.md'

Q3 - used embedder from hw2

In [49]:
from embedder import Embedder

embedder=Embedder()

2026-07-12 15:31:09.663645142 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [54]:
texts=[i["content"] for i in chunks]

In [56]:
X=embedder.encode_batch(texts)

In [57]:
X.shape

(295, 384)

In [58]:
vecindex=VectorSearch()
vecindex.fit(X,chunks)

In [61]:
def vector_search(query, num_results=5):
    que_vector=embedder.encode(query)
    return vecindex.search(que_vector, num_results=num_results)

In [62]:
vec_results= vector_search(q)

In [64]:
vec_results[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

Q4

In [65]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [77]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [68]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [70]:
relevance_total=compute_relevance_total(ground_truth,text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [71]:
def hit_rate(relevance):
    cnt=0

    for i in relevance:
        if 1 in i:
            cnt+=1
    
    return cnt/len(relevance)

In [72]:
hit_rate(relevance_total)

0.7583333333333333

Q5

In [73]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for pos in range(len(line)):
            if line[pos] == 1:
                score=1/(pos+1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [74]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [75]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

Q6

In [78]:
k_results={}
values = [1,50,100,200]

for k in values:
    result = evaluate(
        ground_truth,
        lambda query: hybrid_search(query, k=k)
        )
    
    k_results[k]=result

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [79]:
k_results

{1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449},
 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}